# Data models with XGBoost

In [1]:
import sys, os, joblib
import pyreadstat
import pickle
import ast
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance

In [2]:
# Acuerdate de guardar los datos procesados
file = './trombo_dat2.pkl'
with  open(file,'rb') as fd:
    patD  = pickle.load(fd)
    ldata = pickle.load(fd)
    data  = pickle.load(fd)
#


In [3]:
data.shape

(22874, 16)

In [4]:
patD.shape

(22874, 174)

In [5]:
# Acuerdate de guardar los datos procesados
file2 = './trombo_dat0_nonan.pkl'
with  open(file2,'rb') as fd:
    patDnonan = pickle.load(fd)
#

In [6]:
patDnonan.shape

(119449, 174)

In [7]:
patD.columns

Index(['id_pacie', 'sexo', 'edad', 'raza', 'peso', 'talla', 'tension_',
       'fecha_di', 'f_alta_h', 'ant_inf',
       ...
       'seco', 'secopri', 'f_secop', 'secopres', 'edadC', 'pesoC', 'tensionC',
       'fr_can_eC', 'trb_splcnc', 'othr_TUS'],
      dtype='object', length=174)

In [8]:
cls = [ cl for cl in patD.columns if cl not in ['id_pacie','edad','peso','tension_','talla',
                                                'fecha_di','f_alta_h','f_secop','ana_dura']]
print(cls)

['sexo', 'raza', 'ant_inf', 'ant_isq', 'ant_clau', 'fum_act', 'diabetes', 'hip_art', 'insf_car', 'fibr_aur', 'trat_est', 'e_con_pp', 'e_con_cu', 'e_con_ec', 'e_con_lu', 'e_con_af', 'e_con_be', 'e_con_at', 'e_con_va', 'e_con_ar', 'e_con_ea', 'e_con_pr', 'e_con_ro', 'e_con_av', 'e_con_sm', 'sin_tvp_', 'var171', 'ep_tac', 'ep_tac_r', 'ep_tlss', 'ep_tls', 'ep_tll', 'ep_tlp', 'ep_tlce', 'eptacven', 'ep_t_reg', 'ep_ecoca', 'ep_ec_pr', 'ep_eco_p', 'ep_eco_v', 'var52', 'epecoddv', 'tvp_ecog', 'tvp_eco_', 'tv_l_esu', 'tvp_orig', 'tvpocatt', 'tvpocaef', 'tv_l_ein', 'tvp_prox', 'tv_l_vil', 'tv_l_vfe', 'tv_l_vpp', 'tv_l_vpo', 'tv_l_vme', 'tv_l_ves', 'tv_l_svc', 'tv_l_vre', 'tv_l_vrt', 'tv_l_vrp', 'tv_l_vrn', 'tv_l_vca', 'tv_l_yug', 'tv_l_ova', 'tv_l_sup', 'tv_l_pul', 'tv_l_ove', 'fr_cance', 'fr_can_l', 'fr_can_e', 'metacere', 'fr_cirug', 'fr_inmov', 'fr_inm_m', 'fr_inm_t', 'fr_tvp_a', 'fr_tvp_p', 'fr_antfa', 'fr_antgr', 'fr_tvs_a', 'fr_viaje', 'fr_estro', 'fr_est_m', 'fr_est_v', 'fr_embar', 'fr_em

In [9]:
patD[cls]

,sexo,raza,ant_inf,ant_isq,ant_clau,fum_act,diabetes,hip_art,insf_car,fibr_aur,...,eisq_fol,seco,secopri,secopres,edadC,pesoC,tensionC,fr_can_eC,trb_splcnc,othr_TUS
0,Mujer,Caucásica,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,...,NaN,NaN,NaN,NaN,Senior,Normal,Normal,No,No,No
6,Hombre,Caucásica,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,...,NaN,NaN,NaN,NaN,Senior,Normal,Normal,Sin metástasis,No,No
7,Mujer,Caucásica,NaN,NaN,NaN,NaN,NaN,NaN,Sí,NaN,...,NaN,NaN,NaN,NaN,Senior,Normal,Normal,No,No,No
10,Hombre,Caucásica,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,...,NaN,NaN,NaN,NaN,Senior,Normal,Normal,No,No,No
34,Hombre,Caucásica,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,...,NaN,NaN,NaN,NaN,Medio,Normal,Normal,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119381,Hombre,Caucásica,No,No,No,No,No,Sí,No,No,...,NaN,NaN,NaN,NaN,Medio,Normal,Normal,No,No,No
119393,Mujer,Caucásica,No,No,No,No,No,No,No,No,...,NaN,Sí,Sí,Presencia de trombo,Joven,Normal,Normal,No,No,Sí
119394,Mujer,Romaní,No,No,No,No,No,No,No,No,...,NaN,NaN,NaN,NaN,Joven,Bajo,Normal,No,No,No
119399,Hombre,Caucásica,No,No,No,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Joven,Alto,Baja,No,No,No


In [10]:
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(patD[cls])

In [11]:
mlb.classes_

array(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'C', 'S', 'T',
       'U', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k',
       'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'y',
       'z'], dtype=object)

In [12]:
X.shape

(165, 40)

In [13]:
X

array([[0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [14]:
y = patD['ana_dura'].values
y

['Buscada negativo', 'Buscada negativo', 'Buscada negativo', 'Buscada negativo', 'Buscada negativo', ..., 'Buscada positivo', 'Buscada negativo', 'Buscada negativo', 'Buscada positivo', 'Buscada negativo']
Length: 22874
Categories (3, object): ['Buscada negativo', 'Buscada positivo', 'No buscada']

## Only relevant columns from experts

In [15]:
ncls= ['sexo','edadC','sin_tvp_','trb_splcnc',
             'tv_l_esu','tv_l_svc','othr_TUS','fr_cance',
             'fr_cirug','fr_inmov','fr_tvp_a','fr_estro', # 'ana_port', => Always to No
             'evn_reci','evn_hemo']

In [16]:
XpatD = patD[ncls]

In [17]:
YpatD = patD['ana_dura']

In [18]:
XpatD

,sexo,edadC,sin_tvp_,trb_splcnc,tv_l_esu,tv_l_svc,othr_TUS,fr_cance,fr_cirug,fr_inmov,fr_tvp_a,fr_estro,evn_reci,evn_hemo
0,Mujer,Senior,TVP,No,No,No,No,No,No,No,Sí,No,No,No
6,Hombre,Senior,TVP,No,No,No,No,Sí,No,Sí,Sí,No,No,No
7,Mujer,Senior,TVP,No,No,No,No,No,No,Sí,No,No,No,No
10,Hombre,Senior,TVP/EP,No,No,No,No,No,No,No,No,No,Sí,No
34,Hombre,Medio,EP,No,No,No,No,No,No,No,Sí,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119381,Hombre,Medio,EP,No,No,No,No,No,No,No,No,No,No,No
119393,Mujer,Joven,TVP,No,No,No,Sí,No,No,No,No,Sí,No,Sí
119394,Mujer,Joven,TVP,No,No,Sí,No,No,No,No,No,No,No,No
119399,Hombre,Joven,TVP,No,No,No,No,No,No,No,No,No,No,No


## Modeling

In [19]:

# 2. Definir variable objetivo y características
y_raw = patD['ana_dura'].copy()
X_raw = patD.drop(columns=['ana_dura']).copy()

# 3. Convertimos y_raw a multilabel
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform([[val] for val in y_raw])
label_names = mlb.classes_

# 4. ColumnTransformer con OneHotEncoder para todas las columnas (tratadas como categóricas)
categorical_features = X_raw.columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# 5. Pipeline completo
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', MultiOutputClassifier(HistGradientBoostingClassifier(random_state=42)))
])

In [20]:
Y

array([[1, 0],
       [1, 0],
       [1, 0],
       ...,
       [1, 0],
       [0, 1],
       [1, 0]])

In [21]:
label_names

array(['Buscada negativo', 'Buscada positivo'], dtype=object)

In [22]:
X_raw

,id_pacie,sexo,edad,raza,peso,talla,tension_,fecha_di,f_alta_h,ant_inf,...,seco,secopri,f_secop,secopres,edadC,pesoC,tensionC,fr_can_eC,trb_splcnc,othr_TUS
0,2,Mujer,87,Caucásica,65.0,158,150,2001-03-16,NaN,NaN,...,NaN,NaN,NaN,NaN,Senior,Normal,Normal,No,No,No
6,8,Hombre,77,Caucásica,68.0,162,150,2001-03-11,NaN,NaN,...,NaN,NaN,NaN,NaN,Senior,Normal,Normal,Sin metástasis,No,No
7,9,Mujer,88,Caucásica,54.0,157,140,2001-03-14,NaN,NaN,...,NaN,NaN,NaN,NaN,Senior,Normal,Normal,No,No,No
10,12,Hombre,85,Caucásica,86.0,175,130,2001-03-11,NaN,NaN,...,NaN,NaN,NaN,NaN,Senior,Normal,Normal,No,No,No
34,36,Hombre,64,Caucásica,70.0,165,110,2001-03-19,NaN,NaN,...,NaN,NaN,NaN,NaN,Medio,Normal,Normal,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119381,152926,Hombre,57,Caucásica,77.0,171,143,2022-11-18,2022-11-25,No,...,NaN,NaN,NaN,NaN,Medio,Normal,Normal,No,No,No
119393,152956,Mujer,35,Caucásica,75.0,165,125,2022-11-29,2022-12-10,No,...,Sí,Sí,2024-02-27,Presencia de trombo,Joven,Normal,Normal,No,No,Sí
119394,152957,Mujer,4,Romaní,19.0,-9223372036854775808,110,2024-04-08,2024-04-15,No,...,NaN,NaN,NaN,NaN,Joven,Bajo,Normal,No,No,No
119399,152970,Hombre,43,Caucásica,105.0,176,-9223372036854775808,2022-05-11,2022-05-16,No,...,NaN,NaN,NaN,NaN,Joven,Alto,Baja,No,No,No


In [23]:
# 6. Separar train/test
X_train, X_test, y_train, y_test = train_test_split(X_raw, Y, test_size=0.2, random_state=42)

# 7. GridSearchCV con validación cruzada (cv=5)
param_grid = {
    'classifier__estimator__learning_rate': [0.1, 0.25],
    'classifier__estimator__max_depth': [5,9],
    'classifier__estimator__max_iter': [100]
}

In [24]:

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='f1_micro', verbose=2, n_jobs=1)
grid.fit(X_train, y_train)

# 8. Evaluación
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)


Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=5, classifier__estimator__max_iter=100; total time= 2.9min
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=5, classifier__estimator__max_iter=100; total time= 2.9min
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=5, classifier__estimator__max_iter=100; total time= 3.0min
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=9, classifier__estimator__max_iter=100; total time= 4.5min
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=9, classifier__estimator__max_iter=100; total time= 4.5min
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=9, classifier__estimator__max_iter=100; total time= 4.2min
[CV] END classifier__estimator__learning_rate=0.25, classifier__estimator__max_dep

In [25]:

print("✅ Mejores parámetros encontrados:", grid.best_params_)
print("\n📊 Clasificación por etiqueta de 'ana_dura':")
print(classification_report(y_test, y_pred, target_names=label_names))

✅ Mejores parámetros encontrados: {'classifier__estimator__learning_rate': 0.1, 'classifier__estimator__max_depth': 5, 'classifier__estimator__max_iter': 100}

📊 Clasificación por etiqueta de 'ana_dura':
                  precision    recall  f1-score   support

Buscada negativo       1.00      1.00      1.00      2884
Buscada positivo       1.00      1.00      1.00      1691

       micro avg       1.00      1.00      1.00      4575
       macro avg       1.00      1.00      1.00      4575
    weighted avg       1.00      1.00      1.00      4575
     samples avg       1.00      1.00      1.00      4575



In [26]:
# 9. Guardar el mejor pipeline (preprocesador + modelo)
iter = grid.best_params_['classifier__estimator__max_iter']
lr   = grid.best_params_['classifier__estimator__learning_rate']
depth= grid.best_params_['classifier__estimator__max_depth']
joblib.dump(best_model, f"modelo_xgb_pipeline-{iter}-{lr}-{depth}.pkl")

#    Guardar el binarizador de la variable objetivo
joblib.dump(mlb, f"binarizador_ana_dura-{iter}-{lr}-{depth}.pkl")


['binarizador_ana_dura-100-0.1-5.pkl']

In [27]:
all_labels_proba=best_model.predict_proba(X_test)
all_labels_proba

[array([[0.0025019 , 0.9974981 ],
        [0.01133299, 0.98866701],
        [0.00277186, 0.99722814],
        ...,
        [0.00273345, 0.99726655],
        [0.99733615, 0.00266385],
        [0.98769525, 0.01230475]]),
 array([[0.99746943, 0.00253057],
        [0.98861428, 0.01138572],
        [0.99732218, 0.00267782],
        ...,
        [0.99723516, 0.00276484],
        [0.00295674, 0.99704326],
        [0.01249502, 0.98750498]])]

In [28]:
# Calcular AUC para cada etiqueta
num_labels = y_test.shape[1] # Número total de etiquetas

auc_scores_per_label = []

for i in range(num_labels):
    y_true_label_i = y_test[:, i]
    y_proba_label_i = all_labels_proba[i][:, 1] # Probabilidad de clase 1 para la etiqueta i

    # Solo calcular si la etiqueta tiene al menos una instancia de cada clase (0 y 1) en y_test
    # De lo contrario, roc_auc_score dará un error
    if len(np.unique(y_true_label_i)) == 2:
         auc_i = roc_auc_score(y_true_label_i, y_proba_label_i)
         auc_scores_per_label.append(auc_i)
         print(f"AUC ROC para la etiqueta {i}: {auc_i:.4f}")
    else:
         print(f"Etiqueta {i} tiene solo una clase en y_test, no se puede calcular AUC ROC.")
         # Puedes añadir un valor NaN o similar si quieres mantener el mismo número de entradas que etiquetas


AUC ROC para la etiqueta 0: 0.9992
AUC ROC para la etiqueta 1: 0.9991


In [29]:
y_true_flat = y_test.ravel() # Aplanar y_test
y_proba_combined_positive_class = np.stack([all_labels_proba[i][:, 1] for i in range(num_labels)], axis=1)


In [30]:
# Calcular AUC micro y macro
if num_labels > 1:
    try:
        auc_micro = roc_auc_score(y_test, y_proba_combined_positive_class, average='micro')
        print(f"\nAUC ROC promedio Micro: {auc_micro:.4f}")
    except Exception as e:
        print(f"No se pudo calcular AUC Micro promedio: {e}")

    try:
        # Asegúrate de que todas las etiquetas tengan ambas clases para el promedio macro
        if all(len(np.unique(y_test[:, i])) == 2 for i in range(num_labels)):
             auc_macro = roc_auc_score(y_test, y_proba_combined_positive_class, average='macro')
             print(f"AUC ROC promedio Macro: {auc_macro:.4f}")
        else:
             print("\nNo se puede calcular AUC Macro promedio porque al menos una etiqueta no tiene ambas clases en y_test.")

    except Exception as e:
         print(f"No se pudo calcular AUC Macro promedio: {e}")


AUC ROC promedio Micro: 0.9994
AUC ROC promedio Macro: 0.9992


In [31]:
# 1. Aplicar el preprocesador a X_test para obtener los datos transformados
print("Aplicando preprocesamiento a X_test...")
X_test_preprocessed = best_model.named_steps['preprocessor'].transform(X_test)
print("Preprocesamiento completado.")

# 2. Acceder al MultiOutputClassifier entrenado
multioutput_classifier = best_model.named_steps['classifier']

# 3. Acceder a la lista de clasificadores individuales entrenados (uno por etiqueta)
individual_estimators = multioutput_classifier.estimators_

Aplicando preprocesamiento a X_test...
Preprocesamiento completado.


In [32]:
# 4. Obtener los nombres de las features después del preprocesamiento
try:
    feature_names_out = best_model.named_steps['preprocessor'].get_feature_names_out()
    print(f"Obtenidos {len(feature_names_out)} nombres de features post-procesamiento.")
except AttributeError:
    print("Tu versión de scikit-learn no soporta get_feature_names_out() en el preprocesador.")
    print("Los resultados de importancia se mostrarán con nombres genéricos (feature_0, feature_1, ...).")
    if hasattr(X_test_preprocessed, 'shape'):
         feature_names_out = [f"feature_{i}" for i in range(X_test_preprocessed.shape[1])]
    else:
         print("No se pudo determinar el número de features post-procesamiento.")
         feature_names_out = [] # Lista vacía para evitar errores

Obtenidos 54313 nombres de features post-procesamiento.


In [33]:
# 5. Calcular y mostrar la Importancia por Permutación para cada etiqueta

print("\nImportancia de las Features (por Permutación) por Etiqueta de Salida:")
print("-" * 60)

# Usaremos 'roc_auc' como métrica de scoring, ya que es adecuada para probabilidades binarias
scoring_metric = 'roc_auc'

for i, estimator in enumerate(individual_estimators):
    print(f"\nEtiqueta de Salida {i}:")

    # Extraer los valores reales solo para esta etiqueta
    y_true_label_i = y_test[:, i]

    # Asegurarse de que la etiqueta tiene al menos dos clases en y_true_label_i para calcular AUC
    if len(np.unique(y_true_label_i)) < 2:
        print(f"  Advertencia: La Etiqueta {i} tiene solo una clase en y_test. No se puede calcular la importancia por permutación con '{scoring_metric}'.")
        continue # Pasar a la siguiente etiqueta si no hay ambas clases

    # Calcular la importancia por permutación
    # n_repeats: número de veces que se permuta una feature (para estabilidad)
    # random_state: para reproducibilidad
    # n_jobs: para paralelización (-1 usa todos los cores disponibles)
    print(f"  Calculando importancia por permutación con '{scoring_metric}'...")
    try:
        r = permutation_importance(
            estimator,
            X_test_preprocessed, # Usar datos post-procesamiento
            y_true_label_i,      # Usar valores reales solo para esta etiqueta
            scoring=scoring_metric,
            n_repeats=10,
            random_state=42,
            n_jobs=1
        )

        # Los resultados están en r.importances_mean
        importances_mean = r.importances_mean

        # Combinar nombres de features e importancias medias
        if len(importances_mean) == len(feature_names_out):
             feature_importance_pairs = list(zip(feature_names_out, importances_mean))

             # Ordenar por importancia media en orden descendente
             feature_importance_pairs.sort(key=lambda x: x[1], reverse=True)

             # Mostrar las features más importantes
             num_features_to_show = min(20, len(feature_importance_pairs)) # Mostrar top N
             print(f"  Top {num_features_to_show} features más importantes (mediana de la disminución de '{scoring_metric}'):")
             # Filtrar features con importancia > 0 si hay muchas (opcional)
             # feature_importance_pairs = [(f, imp) for f, imp in feature_importance_pairs if imp > 0]
             for feature, importance in feature_importance_pairs[:num_features_to_show]:
                 print(f"    {feature}: {importance:.4f}")
        else:
            print(f"  Error interno: El número de importancias calculadas ({len(importances_mean)}) no coincide con los nombres de features ({len(feature_names_out)}).")

    except Exception as e:
        print(f"  Error al calcular la importancia por permutación para la Etiqueta {i}: {e}")

print("\n" + "-" * 60)



Importancia de las Features (por Permutación) por Etiqueta de Salida:
------------------------------------------------------------

Etiqueta de Salida 0:
  Calculando importancia por permutación con 'roc_auc'...
  Top 20 features más importantes (mediana de la disminución de 'roc_auc'):
    cat__var160_Sí: 0.0095
    cat__var161_nan: 0.0079
    cat__var157_Sí: 0.0064
    cat__var161_Sí: 0.0043
    cat__var156_Sí: 0.0035
    cat__var162_Sí: 0.0021
    cat__var155_Sí: 0.0015
    cat__var154_Sí: 0.0011
    cat__andujak2_nan: 0.0011
    cat__var159_Sí: 0.0009
    cat__var158_Sí: 0.0007
    cat__var156_nan: 0.0002
    cat__var162_nan: 0.0002
    cat__talla_-9223372036854775808: 0.0001
    cat__var160_nan: 0.0001
    cat__var154_nan: 0.0001
    cat__var157_nan: 0.0001
    cat__var155_nan: 0.0001
    cat__var158_nan: 0.0001
    cat__anadutip_nan: 0.0001

Etiqueta de Salida 1:
  Calculando importancia por permutación con 'roc_auc'...
  Top 20 features más importantes (mediana de la disminució

In [34]:
X_raw.columns.tolist()

['id_pacie',
 'sexo',
 'edad',
 'raza',
 'peso',
 'talla',
 'tension_',
 'fecha_di',
 'f_alta_h',
 'ant_inf',
 'ant_isq',
 'ant_clau',
 'fum_act',
 'diabetes',
 'hip_art',
 'insf_car',
 'fibr_aur',
 'trat_est',
 'e_con_pp',
 'e_con_cu',
 'e_con_ec',
 'e_con_lu',
 'e_con_af',
 'e_con_be',
 'e_con_at',
 'e_con_va',
 'e_con_ar',
 'e_con_ea',
 'e_con_pr',
 'e_con_ro',
 'e_con_av',
 'e_con_sm',
 'sin_tvp_',
 'var171',
 'ep_tac',
 'ep_tac_r',
 'ep_tlss',
 'ep_tls',
 'ep_tll',
 'ep_tlp',
 'ep_tlce',
 'eptacven',
 'ep_t_reg',
 'ep_ecoca',
 'ep_ec_pr',
 'ep_eco_p',
 'ep_eco_v',
 'var52',
 'epecoddv',
 'tvp_ecog',
 'tvp_eco_',
 'tv_l_esu',
 'tvp_orig',
 'tvpocatt',
 'tvpocaef',
 'tv_l_ein',
 'tvp_prox',
 'tv_l_vil',
 'tv_l_vfe',
 'tv_l_vpp',
 'tv_l_vpo',
 'tv_l_vme',
 'tv_l_ves',
 'tv_l_svc',
 'tv_l_vre',
 'tv_l_vrt',
 'tv_l_vrp',
 'tv_l_vrn',
 'tv_l_vca',
 'tv_l_yug',
 'tv_l_ova',
 'tv_l_sup',
 'tv_l_pul',
 'tv_l_ove',
 'fr_cance',
 'fr_can_l',
 'fr_can_e',
 'metacere',
 'fr_cirug',
 'fr_inmov'

## Modelo Más Corto

In [35]:
ncls= ['sexo','edadC','sin_tvp_','trb_splcnc',
             'tv_l_esu','tv_l_svc','othr_TUS','fr_cance',
             'fr_cirug','fr_inmov','fr_tvp_a','fr_estro', # 'ana_port', => Always to No
             'evn_reci','evn_hemo']

In [36]:
# 2. Definir variable objetivo y características
y_raw2 = patD['ana_dura'].copy()
X_raw2 = patD[ncls].copy()

# 3. Convertimos y_raw a multilabel
mlb2 = MultiLabelBinarizer()
Y2 = mlb2.fit_transform([[val] for val in y_raw2])
label_names2 = mlb2.classes_

# 4. ColumnTransformer con OneHotEncoder para todas las columnas (tratadas como categóricas)
categorical_features2 = X_raw2.columns.tolist()
preprocessor2 = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features2)
    ]
)

# 5. Pipeline completo
pipeline2 = Pipeline(steps=[
    ('preprocessor', preprocessor2),
    ('classifier', MultiOutputClassifier(HistGradientBoostingClassifier(random_state=42)))
])

In [37]:
# 6. Separar train/test
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_raw2, Y2, test_size=0.2, random_state=42)

# 7. GridSearchCV con validación cruzada (cv=5)
param_grid2 = {
    'classifier__estimator__learning_rate': [0.1, 0.25],
    'classifier__estimator__max_depth': [5,9,11],
    'classifier__estimator__max_iter': [100]
}

In [38]:
#
grid2 = GridSearchCV(pipeline2, param_grid2, cv=3, scoring='f1_micro', verbose=2, n_jobs=1)
grid2.fit(X_train2, y_train2)

# 8. Evaluación
best_model2 = grid2.best_estimator_
y_pred2 = best_model2.predict(X_test2)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=5, classifier__estimator__max_iter=100; total time=   0.2s
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=5, classifier__estimator__max_iter=100; total time=   0.2s
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=5, classifier__estimator__max_iter=100; total time=   0.2s
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=9, classifier__estimator__max_iter=100; total time=   0.3s
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=9, classifier__estimator__max_iter=100; total time=   0.2s
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_depth=9, classifier__estimator__max_iter=100; total time=   0.2s
[CV] END classifier__estimator__learning_rate=0.1, classifier__estimator__max_dept

In [39]:
#
print("✅ Mejores parámetros encontrados:", grid2.best_params_)
print("\n📊 Clasificación por etiqueta de 'ana_dura':")
print(classification_report(y_test2, y_pred2, target_names=label_names2))

✅ Mejores parámetros encontrados: {'classifier__estimator__learning_rate': 0.1, 'classifier__estimator__max_depth': 9, 'classifier__estimator__max_iter': 100}

📊 Clasificación por etiqueta de 'ana_dura':
                  precision    recall  f1-score   support

Buscada negativo       0.63      0.99      0.77      2884
Buscada positivo       0.48      0.04      0.08      1691

       micro avg       0.63      0.64      0.63      4575
       macro avg       0.56      0.52      0.43      4575
    weighted avg       0.58      0.64      0.52      4575
     samples avg       0.63      0.64      0.63      4575



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [40]:
# 9. Guardar el mejor pipeline (preprocesador + modelo)
iter2 = grid2.best_params_['classifier__estimator__max_iter']
lr2   = grid2.best_params_['classifier__estimator__learning_rate']
depth2= grid2.best_params_['classifier__estimator__max_depth']
joblib.dump(best_model2, f"modelo_xgb_pipeline-{iter2}-{lr2}-{depth2}.pkl")

#    Guardar el binarizador de la variable objetivo
joblib.dump(mlb2, f"binarizador_ana_dura-{iter2}-{lr2}-{depth2}.pkl")

['binarizador_ana_dura-100-0.1-9.pkl']

In [41]:
all_labels_proba2=best_model2.predict_proba(X_test2)

In [42]:
# Calcular AUC para cada etiqueta
num_labels2 = y_test2.shape[1] # Número total de etiquetas

auc_scores_per_label2 = []

for i in range(num_labels2):
    y_true_label_i2 = y_test2[:, i]
    y_proba_label_i2 = all_labels_proba2[i][:, 1] # Probabilidad de clase 1 para la etiqueta i

    # Solo calcular si la etiqueta tiene al menos una instancia de cada clase (0 y 1) en y_test
    # De lo contrario, roc_auc_score dará un error
    if len(np.unique(y_true_label_i2)) == 2:
         auc_i2 = roc_auc_score(y_true_label_i2, y_proba_label_i2)
         auc_scores_per_label2.append(auc_i2)
         print(f"AUC ROC para la etiqueta {i}: {auc_i2:.4f}")
    else:
         print(f"Etiqueta {i} tiene solo una clase en y_test, no se puede calcular AUC ROC.")
         # Puedes añadir un valor NaN o similar si quieres mantener el mismo número de entradas que etiquetas


AUC ROC para la etiqueta 0: 0.5631
AUC ROC para la etiqueta 1: 0.5663


In [43]:
y_true_flat2 = y_test2.ravel() # Aplanar y_test
y_proba_combined_positive_class2 = np.stack(
            [all_labels_proba2[i][:, 1] for i in range(num_labels2)], axis=1)

In [44]:
# Calcular AUC micro y macro
if num_labels > 1:
    try:
        auc_micro2 = roc_auc_score(y_test2, y_proba_combined_positive_class2, average='micro')
        print(f"\nAUC ROC promedio Micro: {auc_micro2:.4f}")
    except Exception as e:
        print(f"No se pudo calcular AUC Micro promedio: {e}")

    try:
        # Asegúrate de que todas las etiquetas tengan ambas clases para el promedio macro
        if all(len(np.unique(y_test2[:, i])) == 2 for i in range(num_labels2)):
             auc_macro2 = roc_auc_score(y_test2, y_proba_combined_positive_class2, 
                                        average='macro')
             print(f"AUC ROC promedio Macro: {auc_macro2:.4f}")
        else:
             print("\nNo se puede calcular AUC Macro promedio porque al menos una etiqueta no tiene ambas clases en y_test.")

    except Exception as e:
         print(f"No se pudo calcular AUC Macro promedio: {e}")


AUC ROC promedio Micro: 0.6604
AUC ROC promedio Macro: 0.5647


In [45]:
# 1. Aplicar el preprocesador a X_test para obtener los datos transformados
print("Aplicando preprocesamiento a X_test...")
X_test_preprocessed2 = best_model2.named_steps['preprocessor'].transform(X_test2)
print("Preprocesamiento completado.")

# 2. Acceder al MultiOutputClassifier entrenado
multioutput_classifier2 = best_model2.named_steps['classifier']

# 3. Acceder a la lista de clasificadores individuales entrenados (uno por etiqueta)
individual_estimators2 = multioutput_classifier2.estimators_

Aplicando preprocesamiento a X_test...
Preprocesamiento completado.


In [46]:
# 4. Obtener los nombres de las features después del preprocesamiento
try:
    feature_names_out2 = best_model2.named_steps['preprocessor'].get_feature_names_out()
    print(f"Obtenidos {len(feature_names_out2)} nombres de features post-procesamiento.")
except AttributeError:
    print("Tu versión de scikit-learn no soporta get_feature_names_out() en el preprocesador.")
    print("Los resultados de importancia se mostrarán con nombres genéricos (feature_0, feature_1, ...).")
    if hasattr(X_test_preprocessed2, 'shape'):
         feature_names_out2 = [f"feature_{i}" for i in range(X_test_preprocessed2.shape[1])]
    else:
         print("No se pudo determinar el número de features post-procesamiento.")
         feature_names_out2 = [] # Lista vacía para evitar errores

Obtenidos 31 nombres de features post-procesamiento.


In [47]:
# 5. Calcular y mostrar la Importancia por Permutación para cada etiqueta

print("\nImportancia de las Features (por Permutación) por Etiqueta de Salida:")
print("-" * 60)

# Usaremos 'roc_auc' como métrica de scoring, ya que es adecuada para probabilidades binarias
scoring_metric2 = 'roc_auc'

for i, estimator in enumerate(individual_estimators2):
    print(f"\nEtiqueta de Salida {i}:")

    # Extraer los valores reales solo para esta etiqueta
    y_true_label_i2 = y_test2[:, i]

    # Asegurarse de que la etiqueta tiene al menos dos clases en y_true_label_i para calcular AUC
    if len(np.unique(y_true_label_i2)) < 2:
        print(f"  Advertencia: La Etiqueta {i} tiene solo una clase en y_test. No se puede calcular la importancia por permutación con '{scoring_metric}'.")
        continue # Pasar a la siguiente etiqueta si no hay ambas clases

    # Calcular la importancia por permutación
    # n_repeats: número de veces que se permuta una feature (para estabilidad)
    # random_state: para reproducibilidad
    # n_jobs: para paralelización (-1 usa todos los cores disponibles)
    print(f"  Calculando importancia por permutación con '{scoring_metric2}'...")
    try:
        r2 = permutation_importance(
            estimator,
            X_test_preprocessed2, # Usar datos post-procesamiento
            y_true_label_i2,      # Usar valores reales solo para esta etiqueta
            scoring=scoring_metric2,
            n_repeats=10,
            random_state=42,
            n_jobs=1
        )

        # Los resultados están en r.importances_mean
        importances_mean2 = r2.importances_mean

        # Combinar nombres de features e importancias medias
        if len(importances_mean2) == len(feature_names_out2):
             feature_importance_pairs2 = list(zip(feature_names_out2, importances_mean2))

             # Ordenar por importancia media en orden descendente
             feature_importance_pairs2.sort(key=lambda x: x[1], reverse=True)

             # Mostrar las features más importantes
             num_features_to_show2 = min(20, len(feature_importance_pairs2)) # Mostrar top N
             print(f"  Top {num_features_to_show2} features más importantes (mediana de la disminución de '{scoring_metric2}'):")
             # Filtrar features con importancia > 0 si hay muchas (opcional)
             # feature_importance_pairs = [(f, imp) for f, imp in feature_importance_pairs if imp > 0]
             for feature, importance in feature_importance_pairs2[:num_features_to_show2]:
                 print(f"    {feature}: {importance:.4f}")
        else:
            print(f"  Error interno: El número de importancias calculadas ({len(importances_mean2)}) no coincide con los nombres de features ({len(feature_names_out2)}).")

    except Exception as e:
        print(f"  Error al calcular la importancia por permutación para la Etiqueta {i}: {e}")

print("\n" + "-" * 60)


Importancia de las Features (por Permutación) por Etiqueta de Salida:
------------------------------------------------------------

Etiqueta de Salida 0:
  Calculando importancia por permutación con 'roc_auc'...
  Top 20 features más importantes (mediana de la disminución de 'roc_auc'):
    cat__edadC_Joven: 0.0229
    cat__tv_l_esu_No: 0.0097
    cat__othr_TUS_No: 0.0070
    cat__fr_cirug_No: 0.0048
    cat__sin_tvp__TVP: 0.0038
    cat__sexo_Hombre: 0.0031
    cat__fr_inmov_No: 0.0023
    cat__trb_splcnc_No: 0.0020
    cat__evn_hemo_No: 0.0012
    cat__edadC_Medio: 0.0011
    cat__sin_tvp__EP: 0.0010
    cat__evn_reci_No: 0.0010
    cat__edadC_Senior: 0.0008
    cat__trb_splcnc_Sí: 0.0003
    cat__tv_l_esu_Sí: 0.0002
    cat__othr_TUS_Sí: 0.0002
    cat__fr_tvp_a_Sí: 0.0001
    cat__fr_cirug_Sí: 0.0001
    cat__tv_l_svc_Sí: 0.0000
    cat__tv_l_svc_No: -0.0001

Etiqueta de Salida 1:
  Calculando importancia por permutación con 'roc_auc'...
  Top 20 features más importantes (mediana 